# 03 Preprocessing & Feature Scaling

Notebook ini untuk melakukan preprocessing data sebelum modeling.

## Langkah-langkah:
1. Load dataset dari data collection
2. Handle missing values (interpolasi time series)
3. Split features dan target
4. Split train-test data
5. Feature scaling dengan StandardScaler
6. Simpan scaler dan data split untuk modeling

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import pickle
import os

# Import fungsi preprocessing dari script
import sys
sys.path.append('../scripts')
from preprocessing import FEATURES, TARGET, load_dataset, handle_missing, split_features_target, scale_features

## Load Dataset

In [ ]:
# Load dataset
df = load_dataset()

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## Cek Missing Values

In [ ]:
print("Missing values sebelum preprocessing:")
print(df.isnull().sum())

print("\nPersentase missing values:")
print((df.isnull().sum() / len(df) * 100).round(2))

## Handle Missing Values

In [ ]:
# Handle missing values dengan interpolasi linear
df_clean = handle_missing(df)

print("Dataset setelah handling missing values:")
print(f"Shape: {df_clean.shape}")
print(f"\nMissing values setelah preprocessing:")
print(df_clean.isnull().sum())

In [ ]:
# Cek apakah ada data yang di-drop
dropped = len(df) - len(df_clean)
if dropped > 0:
    print(f"\n{dropped} baris di-drop karena missing values yang tidak bisa di-interpolate")
else:
    print("\nTidak ada baris yang di-drop")

## Split Features dan Target

In [ ]:
# Split features dan target
X, y = split_features_target(df_clean)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {FEATURES}")
print(f"Target: {TARGET}")

In [ ]:
# Tampilkan statistik features
print("Features Statistics:")
X.describe().round(2)

In [ ]:
# Tampilkan statistik target
print("Target Statistics:")
print(f"Mean: {y.mean():.2f}")
print(f"Std: {y.std():.2f}")
print(f"Min: {y.min():.2f}")
print(f"Max: {y.max():.2f}")
print(f"\nTarget distribution:")
print(y.describe().round(2))

## Train-Test Split

In [ ]:
# Split train-test (80% train, 20% test)
# Karena ini time series, kita tidak shuffle agar urutan waktu tetap
split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Train set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTrain years: {df_clean.iloc[:split_idx]['Year'].min()} - {df_clean.iloc[:split_idx]['Year'].max()}")
print(f"Test years: {df_clean.iloc[split_idx:]['Year'].min()} - {df_clean.iloc[split_idx:]['Year'].max()}")

## Feature Scaling dengan StandardScaler

In [ ]:
# Scale features
os.makedirs('../models', exist_ok=True)
X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train, X_test, 
    save_path='../models/scaler.pkl'
)

print("Feature scaling completed!")
print(f"Scaler saved to ../models/scaler.pkl")

In [ ]:
# Tampilkan statistik sebelum dan sesudah scaling
print("BEFORE SCALING (Train):")
print(X_train.describe().round(2))

print("\nAFTER SCALING (Train):")
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=FEATURES)
print(X_train_scaled_df.describe().round(2))

In [ ]:
# Verifikasi scaling
print("Verification of scaling:")
print(f"Train mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"Train std (should be ~1): {X_train_scaled.std():.6f}")
print(f"Test mean: {X_test_scaled.mean():.6f}")
print(f"Test std: {X_test_scaled.std():.6f}")

## Simpan Data Split untuk Modeling

In [ ]:
# Simpan data split untuk modeling
data_split = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'X_train_scaled': X_train_scaled,
    'X_test_scaled': X_test_scaled,
    'features': FEATURES,
    'target': TARGET,
    'train_years': df_clean.iloc[:split_idx]['Year'].tolist(),
    'test_years': df_clean.iloc[split_idx:]['Year'].tolist()
}

with open('../models/data_split.pkl', 'wb') as f:
    pickle.dump(data_split, f)

print("Data split saved to ../models/data_split.pkl")

## Simpan Dataset Cleaned

In [ ]:
# Simpan dataset cleaned untuk reference
df_clean.to_csv('../data/processed/dataset_cleaned.csv', index=False)
print("Cleaned dataset saved to ../data/processed/dataset_cleaned.csv")

## Visualisasi Distribusi Target pada Train dan Test

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(y_train, kde=True, color='blue', alpha=0.6)
plt.title('Target Distribution - Train Set')
plt.xlabel('GDP Growth (%)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(y_test, kde=True, color='orange', alpha=0.6)
plt.title('Target Distribution - Test Set')
plt.xlabel('GDP Growth (%)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/eda_outputs/train_test_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Ringkasan Preprocessing

In [ ]:
print("="*60)
print("PREPROCESSING SELESAI")
print("="*60)
print(f"\nOriginal dataset size: {len(df)}")
print(f"After cleaning: {len(df_clean)}")
print(f"Dropped rows: {len(df) - len(df_clean)}")

print(f"\nTrain set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

print(f"\nFeatures ({len(FEATURES)}): {', '.join(FEATURES)}")
print(f"Target: {TARGET}")

print("\nFile yang dibuat:")
print("- ../models/scaler.pkl (StandardScaler untuk features)")
print("- ../models/data_split.pkl (Data train-test split)")
print("- ../data/processed/dataset_cleaned.csv (Dataset cleaned)")
print("- ../data/eda_outputs/train_test_target_distribution.png")

print("\nLangkah selanjutnya:")
print("Jalankan 04_modeling.ipynb untuk training model ML")